# Experiment: VN FK LSTM Walk-Forward on Colab

Objective:
- Run the Vietnam-only Fischer-Krauss research pack on Colab with the updated VN market rules.
- Preserve artifacts in Google Drive and resume cleanly after session interruptions.
- Start with the `fast` preset, then switch to `full` once the pipeline and outputs look correct.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab. Adjust paths manually if needed.')

REPO_URL = 'https://github.com/<your-org>/data_stock_market.git'
WORK_ROOT = Path('/content/drive/MyDrive/fk_lstm_vn_walkforward')
REPO_DIR = WORK_ROOT / 'data_stock_market'
OUTPUT_ROOT = WORK_ROOT / 'artifacts' / 'fk_lstm_vn_pack'
INDEX_OUTPUT = WORK_ROOT / 'artifacts' / 'run_index' / 'vn_walkforward_report.html'
PRESET = 'fast'  # change to 'full' after the fast pass looks correct

WORK_ROOT.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Reusing repo at {REPO_DIR}')

%cd {REPO_DIR}


In [ ]:
%pip install -q pandas matplotlib yfinance beautifulsoup4 pydantic packaging ipython

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))


In [ ]:
cmd = (
    'PYTHONPATH=src/training python3 src/training/scripts/run_vn_fischer_krauss_pack.py '
    f'--preset {PRESET} '
    f'--output-root {OUTPUT_ROOT} '
    f'--index-output {INDEX_OUTPUT} '
    '--skip-existing'
)
print(cmd)
!{cmd}


In [ ]:
import pandas as pd
from IPython.display import FileLink, display

summary_path = INDEX_OUTPUT.with_suffix('.csv')
summary_df = pd.read_csv(summary_path)
display(summary_df)
display(FileLink(str(INDEX_OUTPUT)))


## Notes

- The VN pack uses `long_only`, `forward_horizon_days=2`, and `sell_tax_bps=10.0` by default.
- It also applies VN microstructure defaults from `market_rules.py`, including curated VN profile loading, liquidity filters, and limit-up entry blocking.
- Rerun the notebook cell after disconnects; completed runs are skipped automatically.
